In [8]:
import requests

#Seleccione el recurso descargar
url = "https://en.wikipedia.org/wiki/List_of_Spotify_streaming_records"

response = requests.get(url, headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.102 Safari/537.36"})



In [9]:
response

<Response [200]>

In [11]:
#Paso 3: Transformar el HTML
import pandas as pd
from bs4 import BeautifulSoup

In [13]:
soup = BeautifulSoup(html, "html.parser")

# Buscar tablas en la página
tables = soup.find_all("table", class_="wikitable")

print(f"Número de tablas encontradas: {len(tables)}")


# Leer todas las tablas del HTML
df_list = pd.read_html(html)

# Seleccionamos la primera tabla relevante
df = df_list[0]

df.head()

Número de tablas encontradas: 25


C:\Users\Milena Torres\AppData\Local\Temp\ipykernel_3564\748714271.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_list = pd.read_html(html)


,Rank,Song,Artist(s),Streams (billions),Release date,Ref.
0,1,"""Blinding Lights""",The Weeknd,5.319,29 November 2019,[1]
1,2,"""Shape of You""",Ed Sheeran,4.811,6 January 2017,[2]
2,3,"""Sweater Weather""",The Neighbourhood,4.445,3 December 2012,[3]
3,4,"""Starboy""",The Weeknd and Daft Punk,4.414,21 September 2016,[4]
4,5,"""As It Was""",Harry Styles,4.295,1 April 2022,[5]


In [14]:
#Paso 4: Procesar y limpiar el DataFrame Revisar columnas
df.columns

Index(['Rank', 'Song', 'Artist(s)', 'Streams (billions)', 'Release date',
       'Ref.'],
      dtype='object')

In [15]:
#Limpiar columnas numéricas (eliminar $ y B)


# Ejemplo: limpiar todas las columnas tipo object
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace("B", "", regex=False)
            .str.replace("M", "", regex=False)
        )

In [16]:
#Eliminar filas vacías o sin información
df = df.dropna()
df = df.reset_index(drop=True)

df.head()

,Rank,Song,Artist(s),Streams (billions),Release date,Ref.
0,1,"""linding Lights""",The Weeknd,5.319,29 November 2019,[1]
1,2,"""Shape of You""",Ed Sheeran,4.811,6 January 2017,[2]
2,3,"""Sweater Weather""",The Neighbourhood,4.445,3 December 2012,[3]
3,4,"""Starboy""",The Weeknd and Daft Punk,4.414,21 September 2016,[4]
4,5,"""As It Was""",Harry Styles,4.295,1 April 2022,[5]


In [18]:
#5: Almacenar los datos en SQLite

import sqlite3

conn = sqlite3.connect("spotify_records.db")
cursor = conn.cursor()

# Crear tabla (estructura genérica)
cursor.execute("""
CREATE TABLE IF NOT EXISTS spotify_records (
    id INTEGER PRIMARY KEY AUTOINCREMENT
)
""")

In [ ]:
#Ajustar la tabla a las columnas reales del DataFrame

#Creamos la tabla dinámicamente:

columns_sql = ", ".join([f'"{col}" TEXT' for col in df.columns])

cursor.execute(f"""
CREATE TABLE IF NOT EXISTS spotify_data (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    {columns_sql}
)
""")

In [22]:
#Insertar los datos
placeholders = ", ".join(["?"] * len(df.columns))
columns = ", ".join([f'"{col}"' for col in df.columns])  # protege nombres de columnas

query = f"INSERT INTO spotify_data ({columns}) VALUES ({placeholders})"

data = [tuple(row) for row in df.to_numpy()]

cursor.executemany(query, data)

conn.commit()
conn.close()




In [23]:
#Paso 6: Visualización de los datos
import matplotlib.pyplot as plt